# 6 自定义量化 A8W8 matmul 算子开发并接入 Qwen3-8B

本章是整个启航营的**核心实践环节**，将学习到的所有知识沉淀转化为一次完整的自定义算子开发实战。

实践的环境准备、提交方式可参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

### 本章定位

本章将开发一个自定义的量化 matmul 算子 `QmmCustom`，通过 A8W8 量化（Activation INT8 + Weight INT8）减少计算量和内存占用，从而提升推理性能。这是本次启航营中综合性最强的一章——涵盖了从算子概念理解、Tiling 设计、Kernel 实现到算子编译、接入网络与功能性能测试的全链路。

### 通过实践达成的学习目标

- **理解量化推理原理**：掌握 A8W8 量化方案（INT8 激活 + INT8 权重）的数学原理与 Scale 反量化机制，理解 perChannelScale 与 perTokenScale 的作用。
- **设计 Tiling 数据结构与函数**：根据算子原型定义自行设计 TilingData 结构体，实现多核分块的 Tiling 函数，理解 Cube 计算单元的分块策略。
- **实现 Cube+Vector 双路径 Kernel**：完成 Cube-only（INT32 输出）和 Cube+Vector 协作（反量化 BF16 输出）两条 Kernel 路径的实现，理解昇腾 AI Core 的 Cube 与 Vector 协作编程模型。
- **完成算子编译与 Torch 接入**：编译自定义算子并通过 Torch 接口接入，打通“算子开发 → 框架调用”的链路。
- **测试算子功能与性能，尝试优化迭代**：验证自定义算子的精度正确性，测试单算子与网络性能，并根据测试结果对 Tiling 或 Kernel 进行优化迭代。

### 本章大纲

- 环境准备
- A8W8 量化 matmul 算子概念
- 自定义算子实现
    - **任务一：自行设计tilingdata结构体并实现tiling函数**
    - **任务二：实现A8W8输入，INT32输出的kernel**
    - **任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel**
- 编译自定义算子
    - **任务四：成功编译自定义算子**
- 测试自定义算子功能性能
    - **任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程**
- 自定义算子接入模型
- 模型性能测试
    - **任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程**
- 提交要求
---

## 1 环境准备

定位教程目录、准备运行目录，并把 CANN 环境变量导入当前 Notebook 进程：

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_repo_root():
    repo_roots = []
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_8b'
TUTORIAL_SRC_DIR = TUTORIAL_DIR / 'src'
if str(TUTORIAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_SRC_DIR))
SRC_DIR = TUTORIAL_DIR / 'src' / 'op_custom'

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment.')
print('CANN包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

RECIPE_ROOT = TUTORIAL_DIR / 'src/inference_scripts/recipe_qwen3_8b'
QMM_CUSTOM_DIR = SRC_DIR / 'qmm_custom'

print('教程目录 =', TUTORIAL_DIR)
print('推理模型目录 =', RECIPE_ROOT)
print('算子相关文件目录 =', SRC_DIR)
print('算子实现目录 =', QMM_CUSTOM_DIR)
print('[环境准备] 完成', flush=True)

[环境准备] 开始


CANN包路径 = /home/developer/Ascend/cann-9.0.0


教程目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b
推理模型目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b
算子相关文件目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom
算子实现目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom
[环境准备] 完成


安装所需的依赖：

In [2]:
%pip install -r {SRC_DIR}/requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn

Defaulting to user installation because normal site-packages is not writeable


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


Note: you may need to restart the kernel to use updated packages.


---
## 2 A8W8 量化 matmul 算子概念

### 2.1 什么是 A8W8 量化

A8W8 量化是 LLM 推理中常用的一种量化方案：
- **A（Activation）**：将输入激活值从 BF16/FP16 量化为 **INT8**
- **W（Weight）**：将权重从 BF16/FP16 量化为 **INT8**
- **反量化**：计算结果 `INT8 × INT8 → INT32` 后，通过 Scale 反量化回浮点精度

### 2.2 算子规格

在上一章中，通过分析量化模型的profiling，归纳出了自定义算子的输入输出规格：

| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16，无 → INT32；ND 格式, shape [M, N] |
| 架构 | ascend910b |

**注意**：x2 权重矩阵使用 **FRACTAL_NZ** 格式，这是昇腾 AI Core Cube 单元的高效存储格式，能提升 matmul 计算性能。

### 2.3 Scale 模式

QmmCustom 算子支持两种 Scale：

| Scale 类型 | Shape | 作用 | 
|-----------|-------|------|
| **perChannelScale** | `[N]` | 对每个输出通道单独缩放 |
| **perTokenScale** | `[M]` | 对每行输入单独缩放 |

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

## 3 自定义算子实现

与前面学习过的内容一致，本次实践采用 `<<<>>>` Kernel 直调方式开发上述原型的量化 matmul 算子。直调方式通过 `kernel<<<numBlocks, dynUBufSize, stream>>>(args)` 语法在 Host 侧直接启动 Kernel，适合快速验证算子功能与性能。

本节包含如下三个任务：

### 任务一：自行设计 TilingData 结构体并实现 Tiling 函数

根据算子原型与 Kernel 的实现需求，自行设计 `QmmCustomTilingData` 结构体，并实现 `CalcQmmTiling` 函数。Tiling 函数需根据输入的 M、N、K 维度，结合平台硬件资源（L1/L0A/L0B/UB 内存限制、可用核数），计算多核分块参数。可借助 `platform_ascendc::PlatformAscendCManager` 获取硬件信息。

> **Tiling 设计提示**：
> 
> 设计 TilingData 时需要考虑以下要素：
> 1. **`TCubeTiling`**：CANN Matmul 库提供的标准 Tiling 结构体，包含 M/N/K 的分块大小（`baseM`、`baseN`、`baseK`）、使用的核数（`usedCoreNum`）等字段。可通过 `GetMatmulTiling` API 自动计算。
> 2. **`isPertoken`**：标记是否走反量化路径，用于 Kernel 内部分支选择。
> 3. **`workspaceSize`**：Path 2 中 Cube 写入 INT32 中间结果所需的 GM workspace 大小。
> 4. **多核分块策略**：将 M 维度或 N 维度分配到多个核上并行计算，每个核处理一个分块。分块大小需平衡 L1/L0A/L0B 缓存利用率与多核负载均衡。

### 任务二：实现 A8W8 输入、INT32 输出的 Kernel

实现 `QmmCubeBasicKernel` 类，完成纯 Cube 路径的 INT8 矩阵乘法计算。

计算公式： ```result_int32 = x1_int8 @ x2_int8```

### 任务三：实现 A8W8 输入、经 perChannelScale 和 perTokenScale 反量化输出 BF16 的 Kernel

实现 `QmmPertokenKernel` 类，在 Cube 计算的基础上增加 Vector 反量化路径。

计算公式： ```result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale```

其中 `perChannelScale` 沿 N 维广播，`perTokenScale` 沿 M 维广播，两者均为 FLOAT32 类型。

---

### 执行路径说明

在 Host 侧的 `qmm_custom` 函数中，根据是否传入 `pertoken_scale` 设置 `isPertoken` 标志，Kernel 启动时据此选择执行路径：

| 路径 | 条件 | 计算流程 | 输出类型 |
|------|------|----------|----------|
| **Path 1: Cube-only** | `isPertoken == 0` | INT8 matmul → INT32 | INT32 |
| **Path 2: Cube+Vector** | `isPertoken == 1` | INT8 matmul → INT32 → dequant(scale×pertoken) → BF16 | BF16 |

**Path 1（Cube-only）**：仅使用 AI Core 的 Cube 计算单元，完成 INT8×INT8 矩阵乘法后直接将 INT32 结果写入输出 GM，无 Vector 参与。对应 `QmmCubeBasicKernel` 的实现。

**Path 2（Cube+Vector）**：Cube 单元先完成 INT8 matmul 得到 INT32 中间结果并写入 GM workspace；随后 Vector 单元从 GM 读取 INT32 结果，依次完成 `Cast`（INT32→FLOAT32）、乘以 `perChannelScale` 和 `perTokenScale`、`Cast`（FLOAT32→BF16）等反量化操作，最终将 BF16 结果写入输出 GM。对应 `QmmPertokenKernel` 的实现。

Kernel 入口函数 `qmm_custom_kernel` 使用 `__mix__(1, 2)` 属性声明，表示为每个 Block 分配 1 个 Cube 核和 2 个 Vector 核，使 Cube 与 Vector 可在同一 Kernel 中协同工作。Kernel 内部通过 `tilingData.isPertoken` 判断走哪条路径，两条路径复用相同的 Tiling 结构和 Kernel 启动入口。

Tiling、Kernel、Torch 接口的实现均位于 `qmm_custom.asc` 文件中。文件已给出整体框架与函数签名，请自行完成下方代码中的 TODO 部分，以实现完整的自定义算子：

In [3]:
%%writefile {QMM_CUSTOM_DIR}/qmm_custom.asc
#include "acl/acl_base.h"
#include "kernel_operator.h"
#define ASCENDC_CUBE_ONLY
#include "lib/matmul_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
#include "torch_npu/csrc/core/npu/NPUStream.h"

#include <torch/extension.h>

#include <algorithm>
#include <cstdint>

using namespace AscendC;

using QmmAType = matmul::MatmulType<TPosition::GM, CubeFormat::ND, int8_t>;
using QmmBType = matmul::MatmulType<TPosition::GM, CubeFormat::NZ, int8_t>;
using QmmCType = matmul::MatmulType<TPosition::GM, CubeFormat::ND, int32_t>;

constexpr uint64_t QMM_SYSTEM_WORKSPACE_SIZE = 16ULL * 1024ULL * 1024ULL;

#pragma pack(push, 8)
struct alignas(8) QmmCustomTilingData {
  TCubeTiling cubeTiling;
  uint32_t hasPerTokenScale;
  uint32_t m;
  uint32_t n;
  uint32_t k;
  uint64_t systemWorkspaceSize;
};
#pragma pack(pop)

static uint32_t CeilDivHost(uint32_t value, uint32_t divisor) {
  return (value + divisor - 1U) / divisor;
}

static uint32_t AlignUpHost(uint32_t value, uint32_t alignment) {
  return CeilDivHost(value, alignment) * alignment;
}

static QmmCustomTilingData BuildQmmTiling(uint32_t hasPerTokenScale,
                                          uint32_t m, uint32_t n,
                                          uint32_t k) {
  auto platform = platform_ascendc::PlatformAscendCManager::GetInstance();
  TORCH_CHECK(platform != nullptr, "QmmCustom cannot query AscendC platform");

  const uint32_t coreLimit = platform->GetCoreNumAic();
  TORCH_CHECK(coreLimit > 0, "QmmCustom found no available AI Core");

  uint32_t mDim = 1;
  uint32_t nDim = 1;
  if (m <= 64) {
    nDim = std::min(coreLimit, CeilDivHost(n, 256U));
  } else {
    mDim = std::min(coreLimit, 4U);
    nDim = std::min(5U, coreLimit / mDim);
  }

  const uint32_t singleM = AlignUpHost(CeilDivHost(m, mDim), 16U);
  const uint32_t singleN = AlignUpHost(CeilDivHost(n, nDim), 32U);

  matmul_tiling::MultiCoreMatmulTiling builder(*platform);
  builder.SetDim(static_cast<int32_t>(mDim * nDim));
  builder.SetSingleShape(static_cast<int32_t>(singleM),
                         static_cast<int32_t>(singleN),
                         static_cast<int32_t>(k));
  builder.SetAlignSplit(16, 32, -1);
  builder.SetShape(m, n, k);
  builder.SetOrgShape(m, n, k);
  builder.SetAType(matmul_tiling::TPosition::GM,
                   matmul_tiling::CubeFormat::ND,
                   matmul_tiling::DataType::DT_INT8);
  builder.SetBType(matmul_tiling::TPosition::GM,
                   matmul_tiling::CubeFormat::NZ,
                   matmul_tiling::DataType::DT_INT8);
  builder.SetCType(matmul_tiling::TPosition::GM,
                   matmul_tiling::CubeFormat::ND,
                   matmul_tiling::DataType::DT_INT32);
  builder.SetBias(false);

  QmmCustomTilingData result{};
  TORCH_CHECK(builder.GetTiling(result.cubeTiling) != -1,
              "QmmCustom tiling failed for M=", m, ", N=", n, ", K=", k);
  result.hasPerTokenScale = hasPerTokenScale;
  result.m = m;
  result.n = n;
  result.k = k;
  result.systemWorkspaceSize = QMM_SYSTEM_WORKSPACE_SIZE;
  return result;
}

__aicore__ inline uint32_t CeilDivDevice(uint32_t value, uint32_t divisor) {
  return (value + divisor - 1U) / divisor;
}

class QmmCustomKernel {
public:
  matmul::Matmul<QmmAType, QmmBType, QmmCType> matmul;

  __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale,
                              GM_ADDR perTokenScale, GM_ADDR y,
                              GM_ADDR intermediate,
                              const QmmCustomTilingData &data) {
    tiling = data;
    a.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x1), data.m * data.k);
    b.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x2), data.k * data.n);
    scaleGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(scale), data.n);

    if (data.hasPerTokenScale != 0U) {
      perTokenGm.SetGlobalBuffer(
          reinterpret_cast<__gm__ float *>(perTokenScale), data.m);
      c.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(intermediate),
                        data.m * data.n);
      outputBf16.SetGlobalBuffer(reinterpret_cast<__gm__ bfloat16_t *>(y),
                                 data.m * data.n);
    } else {
      c.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(y),
                        data.m * data.n);
    }
  }

  __aicore__ inline void RunCube() {
    if ASCEND_IS_AIV {
      return;
    }

    uint32_t mStart;
    uint32_t nStart;
    uint32_t rows;
    uint32_t cols;
    GetCoreRange(GetBlockIdx(), mStart, nStart, rows, cols);
    if (rows == 0U || cols == 0U) {
      matmul.End();
      return;
    }

    matmul.SetOrgShape(tiling.m, tiling.n, tiling.k);
    matmul.SetSingleShape(rows, cols, tiling.k);
    matmul.SetTensorA(a[mStart * tiling.k], false);
    matmul.SetTensorB(b[nStart * tiling.k], false);
    matmul.IterateAll(c[mStart * tiling.n + nStart]);
    matmul.End();
  }

  __aicore__ inline void RunVector(TPipe &pipe) {
    if (tiling.hasPerTokenScale == 0U) {
      return;
    }
    if ASCEND_IS_AIC {
      return;
    }

    CrossCoreWaitFlag(0x8);

    const uint32_t pairedAic = GetBlockIdx() / GetTaskRation();
    uint32_t coreMStart;
    uint32_t nStart;
    uint32_t coreRows;
    uint32_t cols;
    GetCoreRange(pairedAic, coreMStart, nStart, coreRows, cols);
    if (coreRows == 0U || cols == 0U) {
      return;
    }

    const uint32_t rowsPerAiv = CeilDivDevice(coreRows, GetTaskRation());
    const uint32_t mStart = coreMStart + GetSubBlockIdx() * rowsPerAiv;
    const uint32_t coreMEnd = coreMStart + coreRows;
    const uint32_t mEnd =
        (mStart + rowsPerAiv < coreMEnd) ? mStart + rowsPerAiv : coreMEnd;
    if (mStart >= mEnd) {
      return;
    }

    constexpr uint32_t vectorTile = 4096;
    TQue<TPosition::VECIN, 1> intQueue;
    TQue<TPosition::VECIN, 1> scaleQueue;
    TBuf<TPosition::VECCALC> floatBuffer;
    TQue<TPosition::VECOUT, 1> outputQueue;
    pipe.InitBuffer(intQueue, 1, vectorTile * sizeof(int32_t));
    pipe.InitBuffer(scaleQueue, 1, vectorTile * sizeof(float));
    pipe.InitBuffer(floatBuffer, vectorTile * sizeof(float));
    pipe.InitBuffer(outputQueue, 1, vectorTile * sizeof(bfloat16_t));

    for (uint32_t row = mStart; row < mEnd; ++row) {
      const float tokenScale = perTokenGm.GetValue(row);
      for (uint32_t col = 0; col < cols; col += vectorTile) {
        const uint32_t count =
            (cols - col < vectorTile) ? cols - col : vectorTile;
        const uint32_t offset = row * tiling.n + nStart + col;

        LocalTensor<int32_t> intLocal = intQueue.AllocTensor<int32_t>();
        LocalTensor<float> scaleLocal = scaleQueue.AllocTensor<float>();
        DataCopy(intLocal, c[offset], count);
        DataCopy(scaleLocal, scaleGm[nStart + col], count);
        intQueue.EnQue(intLocal);
        scaleQueue.EnQue(scaleLocal);

        intLocal = intQueue.DeQue<int32_t>();
        scaleLocal = scaleQueue.DeQue<float>();
        LocalTensor<float> floatLocal = floatBuffer.Get<float>();
        LocalTensor<bfloat16_t> outLocal =
            outputQueue.AllocTensor<bfloat16_t>();

        Cast(floatLocal, intLocal, RoundMode::CAST_NONE, count);
        PipeBarrier<PIPE_V>();
        Muls(scaleLocal, scaleLocal, tokenScale, count);
        PipeBarrier<PIPE_V>();
        Mul(floatLocal, floatLocal, scaleLocal, count);
        PipeBarrier<PIPE_V>();
        Cast(outLocal, floatLocal, RoundMode::CAST_RINT, count);

        outputQueue.EnQue(outLocal);
        intQueue.FreeTensor(intLocal);
        scaleQueue.FreeTensor(scaleLocal);
        outLocal = outputQueue.DeQue<bfloat16_t>();
        DataCopy(outputBf16[offset], outLocal, count);
        outputQueue.FreeTensor(outLocal);
      }
    }
  }

private:
  __aicore__ inline void GetCoreRange(uint32_t core, uint32_t &mStart,
                                      uint32_t &nStart, uint32_t &rows,
                                      uint32_t &cols) const {
    const uint32_t mBlocks =
        CeilDivDevice(tiling.m, tiling.cubeTiling.singleCoreM);
    const uint32_t nBlocks =
        CeilDivDevice(tiling.n, tiling.cubeTiling.singleCoreN);

    uint32_t mBlock;
    uint32_t nBlock;
    if (tiling.cubeTiling.iterateOrder == 0) {
      mBlock = core % mBlocks;
      nBlock = core / mBlocks;
    } else {
      mBlock = core / nBlocks;
      nBlock = core % nBlocks;
    }

    mStart = mBlock * tiling.cubeTiling.singleCoreM;
    nStart = nBlock * tiling.cubeTiling.singleCoreN;
    if (mStart >= tiling.m || nStart >= tiling.n) {
      rows = 0;
      cols = 0;
      return;
    }

    rows = (mStart + tiling.cubeTiling.singleCoreM <= tiling.m)
               ? tiling.cubeTiling.singleCoreM
               : tiling.m - mStart;
    cols = (nStart + tiling.cubeTiling.singleCoreN <= tiling.n)
               ? tiling.cubeTiling.singleCoreN
               : tiling.n - nStart;
  }

  GlobalTensor<int8_t> a;
  GlobalTensor<int8_t> b;
  GlobalTensor<int32_t> c;
  GlobalTensor<float> scaleGm;
  GlobalTensor<float> perTokenGm;
  GlobalTensor<bfloat16_t> outputBf16;
  QmmCustomTilingData tiling;
};

__global__ __mix__(1, 2) __aicore__ void qmm_custom_kernel(
    GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR perTokenScale, GM_ADDR y,
    GM_ADDR intermediate, __kfc_workspace__ GM_ADDR systemWorkspace,
    QmmCustomTilingData tiling) {
  if (systemWorkspace == nullptr) {
    return;
  }

  TPipe pipe;
  QmmCustomKernel kernel;
  if ASCEND_IS_AIC {
    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), kernel.matmul,
                      &tiling.cubeTiling);
  }
  kernel.Init(x1, x2, scale, perTokenScale, y, intermediate, tiling);
  kernel.RunCube();
  if (tiling.hasPerTokenScale != 0U) {
    if ASCEND_IS_AIC {
      CrossCoreSetFlag<0x2, PIPE_FIX>(0x8);
    }
  }
  kernel.RunVector(pipe);
  pipe.Destroy();
}

namespace ascendc_ops {

at::Tensor qmm_custom(const at::Tensor &x1, const at::Tensor &x2,
                      const at::Tensor &scale,
                      const c10::optional<at::Tensor> &perTokenScale) {
  TORCH_CHECK(x1.dim() == 2 && x2.dim() == 2,
              "QmmCustom expects rank-2 x1 and x2");
  TORCH_CHECK(x1.scalar_type() == at::ScalarType::Char &&
                  x2.scalar_type() == at::ScalarType::Char,
              "QmmCustom expects INT8 x1 and x2");
  TORCH_CHECK(scale.scalar_type() == at::ScalarType::Float,
              "QmmCustom expects FLOAT32 scale");

  const int64_t m = x1.size(0);
  const int64_t k = x1.size(1);
  const int64_t n = x2.size(1);
  TORCH_CHECK(x2.size(0) == k, "QmmCustom K dimensions do not match");
  TORCH_CHECK(scale.numel() == n, "QmmCustom scale length must equal N");
  TORCH_CHECK(m > 0 && n > 0 && k > 0 &&
                  m <= static_cast<int64_t>(UINT32_MAX) &&
                  n <= static_cast<int64_t>(UINT32_MAX) &&
                  k <= static_cast<int64_t>(UINT32_MAX),
              "QmmCustom received an unsupported shape");

  const uint32_t hasPerTokenScale =
      perTokenScale.has_value() && perTokenScale.value().defined() ? 1U : 0U;
  if (hasPerTokenScale != 0U) {
    TORCH_CHECK(perTokenScale.value().scalar_type() == at::ScalarType::Float,
                "QmmCustom expects FLOAT32 per-token scale");
    TORCH_CHECK(perTokenScale.value().numel() == m,
                "QmmCustom per-token scale length must equal M");
  }

  const at::ScalarType outputType = hasPerTokenScale != 0U
                                        ? at::ScalarType::BFloat16
                                        : at::ScalarType::Int;
  at::Tensor output = at::empty({m, n}, x1.options().dtype(outputType));
  at::Tensor intermediate = at::empty(
      {hasPerTokenScale != 0U ? m * n : 1},
      x1.options().dtype(at::ScalarType::Int));

  const QmmCustomTilingData tiling = BuildQmmTiling(
      hasPerTokenScale, static_cast<uint32_t>(m), static_cast<uint32_t>(n),
      static_cast<uint32_t>(k));
  at::Tensor systemWorkspace = at::empty(
      {static_cast<int64_t>(tiling.systemWorkspaceSize)},
      x1.options().dtype(at::ScalarType::Byte));

  auto stream = c10_npu::getCurrentNPUStream().stream(true);
  uint8_t *perTokenAddress =
      hasPerTokenScale != 0U
          ? reinterpret_cast<uint8_t *>(
                perTokenScale.value().mutable_data_ptr())
          : nullptr;
  qmm_custom_kernel<<<tiling.cubeTiling.usedCoreNum, 0, stream>>>(
      reinterpret_cast<uint8_t *>(x1.mutable_data_ptr()),
      reinterpret_cast<uint8_t *>(x2.mutable_data_ptr()),
      reinterpret_cast<uint8_t *>(scale.mutable_data_ptr()), perTokenAddress,
      reinterpret_cast<uint8_t *>(output.mutable_data_ptr()),
      reinterpret_cast<uint8_t *>(intermediate.mutable_data_ptr()),
      reinterpret_cast<uint8_t *>(systemWorkspace.mutable_data_ptr()), tiling);
  return output;
}

} // namespace ascendc_ops

TORCH_LIBRARY(ascendc_ops, m) {
  m.def("qmm_custom(Tensor x1, Tensor x2, Tensor scale, Tensor? "
        "pertoken_scale=None) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m) {
  m.impl("qmm_custom", TORCH_FN(ascendc_ops::qmm_custom));
}


Overwriting /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc


---
## 4 编译自定义算子

### 任务四：成功编译自定义算子


在算子目录下执行编译命令：

In [4]:
!cd {QMM_CUSTOM_DIR} && mkdir -p build && cd build && cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && make -j;

CMake Warning at /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:22 (message):
  static library kineto_LIBRARY-NOTFOUND not found.
Call Stack (most recent call first):
  /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:125 (append_torchlib_if_found)
  CMakeLists.txt:14 (find_package)




-- Configuring done


-- Generating done
-- Build files have been written to: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/build
Consolidate compiler generated dependencies of target ascendc_ops


[ 50%] Building ASC object CMakeFiles/ascendc_ops.dir/qmm_custom.asc.o


[100%] Linking ASC shared library libascendc_ops.so


[100%] Built target ascendc_ops


编译成功后会在build目录下生成ascendc_ops.so

---
## 5 测试自定义算子功能性能

### 任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程

使用测试脚本验证 QmmCustom 算子在不同规格输入下的精度和性能。测试脚本覆盖多组 (M, K, N) 规格，分别验证 INT32 输出（无 perTokenScale）和 BF16 输出（有 perTokenScale）两种路径。

核心接口调用如下：

```python
output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale_npu)
```

### 5.1 测试算子功能

需要测试的 cases 的 shape 如下，对应的是 Qwen3-8B 模型中实际出现的调用，**全部通过**视为算子功能OK：
| M    | K     | N     |
| ---- | ----- | ----- |
| 1    | 4096  | 4096  |
| 1    | 4096  | 6144  |
| 1    | 4096  | 24576 |
| 1    | 12288 | 4096  |
| 50   | 4096  | 4096  |
| 50   | 4096  | 6144  |
| 50   | 4096  | 24576 |
| 50   | 12288 | 4096  |
| 4096 | 4096  | 4096  |
| 4096 | 4096  | 6144  |
| 4096 | 4096  | 24576 |
| 4096 | 12288 | 4096  |

In [5]:
import gc
import torch
import torch_npu

LIB_PATH = QMM_CUSTOM_DIR / "build/libascendc_ops.so"
torch.ops.load_library(str(LIB_PATH))
torch.npu.config.allow_internal_format = True
torch.npu.config.allow_hf32 = False
torch.npu.set_compile_mode(jit_compile=False)
torch.manual_seed(252)

TEST_SHAPES = [
    (1, 4096, 4096), (1, 4096, 6144), (1, 4096, 24576),
    (1, 12288, 4096), (50, 4096, 4096), (50, 4096, 6144),
    (50, 4096, 24576), (50, 12288, 4096),
    (4096, 4096, 4096), (4096, 4096, 6144),
    (4096, 4096, 24576), (4096, 12288, 4096),
]

def make_inputs(M, K, N):
    x1 = torch.randint(-128, 127, (M, K), dtype=torch.int8, device="npu")
    weight_nd = torch.randint(-128, 127, (K, N), dtype=torch.int8, device="npu")
    x2 = torch_npu.npu_format_cast(weight_nd, 29)
    scale = torch.randn(N, dtype=torch.float32, device="npu")
    pertoken_scale = torch.randn(M, dtype=torch.float32, device="npu")
    return x1, weight_nd, x2, scale, pertoken_scale

def test_shape(M, K, N):
    x1, weight_nd, x2, scale, pertoken_scale = make_inputs(M, K, N)
    expected_int32 = torch_npu.npu_quant_matmul(
        x1, x2, scale, pertoken_scale=None, bias=None,
        output_dtype=torch.int32)
    actual_int32 = torch.ops.ascendc_ops.qmm_custom(x1, x2, scale, None)
    expected_bf16 = torch_npu.npu_quant_matmul(
        x1, x2, scale, pertoken_scale=pertoken_scale,
        bias=None, output_dtype=torch.bfloat16)
    actual_bf16 = torch.ops.ascendc_ops.qmm_custom(
        x1, x2, scale, pertoken_scale)
    torch.npu.synchronize()

    int32_mismatch = int((actual_int32 != expected_int32).sum().cpu())
    bf16_mismatch = int((actual_bf16 != expected_bf16).sum().cpu())
    max_abs_error = float(
        (actual_bf16.float() - expected_bf16.float()).abs().max().cpu())
    int32_pass = int32_mismatch == 0
    bf16_pass = torch.allclose(
        actual_bf16.float(), expected_bf16.float(), rtol=0.01, atol=0.01)

    del x1, weight_nd, x2, scale, pertoken_scale
    del expected_int32, actual_int32, expected_bf16, actual_bf16
    gc.collect()
    torch.npu.empty_cache()
    return int32_pass, bf16_pass, int32_mismatch, bf16_mismatch, max_abs_error

results = []
for index, (M, K, N) in enumerate(TEST_SHAPES, 1):
    int32_pass, bf16_pass, int_mis, bf_mis, max_abs = test_shape(M, K, N)
    results.append((M, K, N, int32_pass, bf16_pass))
    print(
        f"[{index:02d}/12] M={M}, K={K}, N={N} | "
        f"INT32={'PASS' if int32_pass else 'FAIL'} mismatch={int_mis} | "
        f"BF16={'PASS' if bf16_pass else 'FAIL'} mismatch={bf_mis} "
        f"max_abs={max_abs:g}",
        flush=True,
    )

passed = sum(int(p1) + int(p2) for _, _, _, p1, p2 in results)
print(f"功能测试汇总: {passed}/24 PASS")
assert passed == 24, f"QmmCustom 功能测试未全部通过: {passed}/24"


/home/developer/.local/lib/python3.11/site-packages/torch_npu/__init__.py:320: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


[01/12] M=1, K=4096, N=4096 | INT32=PASS mismatch=0 | BF16=PASS mismatch=0 max_abs=0


[02/12] M=1, K=4096, N=6144 | INT32=PASS mismatch=0 | BF16=PASS mismatch=0 max_abs=0


[03/12] M=1, K=4096, N=24576 | INT32=PASS mismatch=0 | BF16=PASS mismatch=0 max_abs=0


[04/12] M=1, K=12288, N=4096 | INT32=PASS mismatch=0 | BF16=PASS mismatch=0 max_abs=0


...

[09/12] M=4096, K=4096, N=4096 | INT32=PASS mismatch=0 | BF16=PASS mismatch=92 max_abs=8192


[10/12] M=4096, K=4096, N=6144 | INT32=PASS mismatch=0 | BF16=PASS mismatch=137 max_abs=8192


[11/12] M=4096, K=4096, N=24576 | INT32=PASS mismatch=0 | BF16=PASS mismatch=546 max_abs=16384


[12/12] M=4096, K=12288, N=4096 | INT32=PASS mismatch=0 | BF16=PASS mismatch=87 max_abs=16384


功能测试汇总: 24/24 PASS


### 5.2 测试算子性能
基于torch_npu.profiler来测试算子性能，执行下面的code cell测试单算子性能数据：

In [6]:
import gc
import time
from pathlib import Path
import pandas as pd
import torch_npu

PROF_OUTPUT_DIR = SRC_DIR / "prof_output" / time.strftime("%Y%m%d_%H%M%S")
PROF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
profiler = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.NPU,
        torch_npu.profiler.ProfilerActivity.CPU,
    ],
    experimental_config=torch_npu.profiler._ExperimentalConfig(
        profiler_level=torch_npu.profiler.ProfilerLevel.Level1,
        aic_metrics=torch_npu.profiler.AiCMetrics.PipeUtilization,
    ),
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler(
        str(PROF_OUTPUT_DIR)),
)

profiler.start()
for M, K, N in TEST_SHAPES:
    x1, weight_nd, x2, scale, pertoken_scale = make_inputs(M, K, N)
    torch.ops.ascendc_ops.qmm_custom(x1, x2, scale, None)
    torch.ops.ascendc_ops.qmm_custom(x1, x2, scale, pertoken_scale)
    torch.npu.synchronize()
    profiler.step()
    del x1, weight_nd, x2, scale, pertoken_scale
    gc.collect()
    torch.npu.empty_cache()
profiler.stop()

kernel_csvs = sorted(PROF_OUTPUT_DIR.rglob("kernel_details.csv"))
assert kernel_csvs, f"未在 {PROF_OUTPUT_DIR} 找到 kernel_details.csv"
kernel_csv = kernel_csvs[-1]
kernel_df = pd.read_csv(kernel_csv)
name_column = kernel_df.get("Name", pd.Series("", index=kernel_df.index))
qmm_df = kernel_df[
    name_column.fillna("").str.contains("qmm_custom_kernel", case=False)
].copy()
assert not qmm_df.empty, "Profiler 中未找到 qmm_custom_kernel"
duration_column = next(
    column for column in qmm_df.columns if column.lower().startswith("duration"))
print("单算子 Profiler:", kernel_csv)
print("QmmCustom kernel 调用数:", len(qmm_df))
print("平均耗时(us):", float(qmm_df[duration_column].mean()))
print("最小/最大耗时(us):",
      float(qmm_df[duration_column].min()),
      float(qmm_df[duration_column].max()))
display(qmm_df[["Name", duration_column]].head(24))


[2026-07-23 19:50:23] [WARNING] [204995] profiler.py: Incorrect schedule: Stop profiler while current state is RECORD which may result in incomplete parsed data.


[2026-07-23 19:50:24] [INFO] [207105] profiler.py: Start parsing profiling data in sync mode at: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/prof_output/20260723_195021/bac864229d534783a9c704279102a44b_204995_20260723195021437_ascend_pt

|

/

-

[2026-07-23 19:50:26] [INFO] [207137] profiler.py: CANN profiling data parsed in a total time of 0:00:02.018582

[2026-07-23 19:50:27] [INFO] [207105] profiler.py: All profiling data parsed in a total time of 0:00:03.013314

单算子 Profiler: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/prof_output/20260723_195021/bac864229d534783a9c704279102a44b_204995_20260723195021437_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
QmmCustom kernel 调用数: 24
平均耗时(us): 530.7757499999999
最小/最大耗时(us): 17.74 3437.868


,Name,Duration(us)
7,_Z17qmm_custom_kernelPhS_S_S_S_S_S_19QmmCustom...,17.920
8,_Z17qmm_custom_kernelPhS_S_S_S_S_S_19QmmCustom...,19.401
16,_Z17qmm_custom_kernelPhS_S_S_S_S_S_19QmmCustom...,24.041
17,_Z17qmm_custom_kernelPhS_S_S_S_S_S_19QmmCustom...,28.161
25,_Z17qmm_custom_kernelPhS_S_S_S_S_S_19QmmCustom...,83.402
26,_Z17qmm_custom_kernelPhS_S_S_S_S_S_19QmmCustom...,60.521
34,_Z17qmm_custom_kernelPhS_S_S_S_S_S_19QmmCustom...,34.001
35,_Z17qmm_custom_kernelPhS_S_S_S_S_S_19QmmCustom...,35.501
43,_Z17qmm_custom_kernelPhS_S_S_S_S_S_19QmmCustom...,17.740
44,_Z17qmm_custom_kernelPhS_S_S_S_S_S_19QmmCustom...,26.521


---
## 6 自定义算子接入模型

### 6.1 定位替换点

Qwen3-8B 推理框架中的 W8A8 量化 matmul 通过 `CompressedTensorsW8A8Int8LinearMethod` 类实现，其 `apply_weights` 方法中使用 `torch_npu.npu_quant_matmul` 执行 INT8 量化矩阵乘法。我们需要将这个调用替换为自定义的 `torch.ops.ascendc_ops.qmm_custom` 算子。

查看原始实现：

In [7]:
!grep -n 'npu_quant_matmul' {RECIPE_ROOT}/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py

105:            x = torch_npu.npu_quant_matmul(


### 6.2 替换实现

在 `CompressedTensorsW8A8Int8LinearMethod` 中，将 `torch_npu.npu_quant_matmul` 替换为 `torch.ops.ascendc_ops.qmm_custom`。

替换后的方法为：

```python
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

def apply_weights(self, layer: torch.nn.Module,
                x: torch.Tensor,
                bias: Optional[torch.Tensor],
                dynamic_scale: Optional = None,
                out_dtype: Optional = torch.bfloat16
                ) -> Union[torch.Tensor, Dict[str, Any]]:
    if dynamic_scale is not None:
        x_scale = dynamic_scale
    else:
        x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)
    out_shape_dim_0 = x.size()[:-1]
    x = x.view(-1, x.size(-1))
    x_scale = x_scale.view(-1)

    # --- 替换核心: npu_quant_matmul → qmm_custom ---
    x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight, 
                                layer.weight_scale.view(-1),
                                pertoken_scale=None if out_dtype == torch.int32 else x_scale)
    # --- 替换结束 ---

    x = x.view(out_shape_dim_0 + (-1, ))
    if out_dtype == torch.int32:
        return x, x_scale
    else:
        return x
```

执行如下操作替换推理网络中调用的接口实现：

In [8]:
# 6.2 替换实现: 通过环境变量在内置算子和 QmmCustom 之间切换
import os
from pathlib import Path

W8A8_FILE = (
    RECIPE_ROOT
    / "module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py"
)
code = W8A8_FILE.read_text(encoding="utf-8")

if "_QMM_CUSTOM_LIBRARY = os.environ.get" not in code:
    code = code.replace(
        "from typing import Callable, Any, Dict, List, Optional, Union\n",
        "import os\nfrom typing import Callable, Any, Dict, List, Optional, Union\n",
        1,
    )
    code = code.replace(
        "import torch_npu\n",
        'import torch_npu\n\n'
        '_QMM_CUSTOM_LIBRARY = os.environ.get("QMM_CUSTOM_LIBRARY")\n'
        'if _QMM_CUSTOM_LIBRARY:\n'
        '    torch.ops.load_library(_QMM_CUSTOM_LIBRARY)\n',
        1,
    )
    old_call = """        x = torch_npu.npu_quant_matmul(x, layer.weight,
                                    layer.weight_scale.view(-1),
                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,
                                    bias=layer.bias,
                                    output_dtype=out_dtype)"""
    new_call = """        if _QMM_CUSTOM_LIBRARY:
            x = torch.ops.ascendc_ops.qmm_custom(
                x, layer.weight, layer.weight_scale.view(-1),
                pertoken_scale=None if out_dtype == torch.int32 else x_scale)
            if layer.bias is not None:
                x = x + layer.bias
        else:
            x = torch_npu.npu_quant_matmul(
                x, layer.weight, layer.weight_scale.view(-1),
                pertoken_scale=None if out_dtype == torch.int32 else x_scale,
                bias=layer.bias, output_dtype=out_dtype)"""
    assert old_call in code, "未找到官方 npu_quant_matmul 调用，无法完成接入"
    code = code.replace(old_call, new_call, 1)
    W8A8_FILE.write_text(code, encoding="utf-8")
    print("已加入 QMM_CUSTOM_LIBRARY 切换逻辑")
else:
    print("QMM_CUSTOM_LIBRARY 切换逻辑已存在")

QMM_CUSTOM_LIBRARY = str(QMM_CUSTOM_DIR / "build/libascendc_ops.so")
assert Path(QMM_CUSTOM_LIBRARY).exists(), QMM_CUSTOM_LIBRARY
os.environ["QMM_CUSTOM_LIBRARY"] = QMM_CUSTOM_LIBRARY
print("自定义算子库:", QMM_CUSTOM_LIBRARY)


QMM_CUSTOM_LIBRARY 切换逻辑已存在
自定义算子库: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/build/libascendc_ops.so


### 6.3 替换实现后执行量化推理

指定本地的量化模型权重路径，调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，对比替换前的生成文本是否一致。

**注意：需要先完成第五章的量化模型推理过程，并将`W8A8_WEIGHT`指定为本地的量化模型权重的路径**

In [9]:
import os
import subprocess
import sys
from pathlib import Path
from inference_scripts.recipe_workflow import prepare_runtime_yaml

default_weight = TUTORIAL_DIR / "src/data/models/Qwen3-8B-W8A8"
W8A8_WEIGHT = Path(
    os.environ.get("QWEN3_8B_W8A8_MODEL_PATH", str(default_weight))
).expanduser().resolve()
assert W8A8_WEIGHT.exists(), (
    f"W8A8 权重不存在: {W8A8_WEIGHT}. "
    "请设置 QWEN3_8B_W8A8_MODEL_PATH。"
)

YAML_A8W8_TEMPLATE = (
    RECIPE_ROOT / "models/qwen/config/qwen3_8b_a8w8_1tp.yaml"
)
YAML_A8W8_CUSTOM = (
    RECIPE_ROOT / "models/qwen/config/qwen3_8b_a8w8_custom_submission.yaml"
)
prepare_runtime_yaml(
    YAML_A8W8_TEMPLATE, YAML_A8W8_CUSTOM, model_path=str(W8A8_WEIGHT)
)
env = os.environ.copy()
env["QMM_CUSTOM_LIBRARY"] = QMM_CUSTOM_LIBRARY
env["PATH"] = os.path.dirname(sys.executable) + ":" + env.get("PATH", "")

def print_inference_summary(output):
    lines = output.splitlines()
    for line in lines:
        if (
            "get env QMM_CUSTOM_LIBRARY" in line
            or "Inference time (prefill)" in line
            or "decode average inference time cost" in line
        ):
            print(line)

    for index, line in enumerate(lines):
        marker = "Request 0: outputs:"
        if marker not in line:
            continue
        generated = [line.split(marker, 1)[1].lstrip()]
        for following in lines[index + 1:]:
            if following.startswith("[INFO]") or following.startswith("[ERROR]"):
                break
            generated.append(following)
        print("\n生成文本:\n" + "\n".join(generated).strip())
        break

def run_recipe(yaml_path):
    result = subprocess.run(
        [
            "bash", "executor/scripts/infer.sh", "--model", "qwen",
            "--mode", "offline", "--yaml", str(yaml_path),
        ],
        cwd=str(RECIPE_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if result.returncode != 0:
        print("\n".join(result.stdout.splitlines()[-100:]))
        result.check_returncode()
    print_inference_summary(result.stdout)
    print("推理完成，returncode =", result.returncode)
    return result

print("量化模型:", W8A8_WEIGHT)
result = run_recipe(YAML_A8W8_CUSTOM)


量化模型: /mnt/workspace/models/Qwen3-8B-W8A8-amct


[INFO][2026-07-23 19:50:36.302][patch_getenv.py:15] get env QMM_CUSTOM_LIBRARY = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/build/libascendc_ops.so
[INFO][2026-07-23 19:51:00.806][scheduler.py:518] [Main] Inference time (prefill): 44.95 ms
[INFO][2026-07-23 19:51:11.678][infer.py:127] qwen3_8b decode average inference time cost is 40.84 ms

生成文本:
The output of an attention function is a **weighted sum of the value vectors**, where the weights are determined by the similarity between the **query vector** and each **key vector** in the set of key-value pairs.

### Mathematically, the attention function can be described as:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Where:
- $ Q $ is the **query matrix** (shape: $ [n \times d_k] $)
- $ K $ is the **key matrix** (shape: $ [m \times d_k] $)
- $ V $ is the **value matrix** (shape: $ [m \times d_v] $)
- $ d_k $ is the dimension of the key

---
## 7 模型性能测试

### 任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程

将 QmmCustom 算子接入模型后，使用 Profiling 工具观测模型推理性能，对比自定义算子替换前后的 matmul 耗时变化。

### 7.1 准备 Profiling YAML

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将 YAML 配置中的enable_profiler配置为 True ：

In [10]:
from inference_scripts.recipe_workflow import prepare_profiling_yaml

YAML_A8W8_CUSTOM_PROFILER = (
    RECIPE_ROOT
    / "models/qwen/config/qwen3_8b_a8w8_custom_submission_profiler.yaml"
)
prepare_profiling_yaml(
    YAML_A8W8_CUSTOM,
    YAML_A8W8_CUSTOM_PROFILER,
    enable_profiler=True,
)
print("Profiler YAML:", YAML_A8W8_CUSTOM_PROFILER)


Profiler YAML: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom_submission_profiler.yaml


### 7.2 启动 Profiling 推理

使用 recipes 命令启动带 Profiler 的推理，推理结束后会在结果目录中生成 `prof/` 产物。

In [11]:
result = run_recipe(YAML_A8W8_CUSTOM_PROFILER)
print("Profiler 推理完成")


[INFO][2026-07-23 19:51:27.544][patch_getenv.py:15] get env QMM_CUSTOM_LIBRARY = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/build/libascendc_ops.so
[INFO][2026-07-23 19:51:56.171][scheduler.py:518] [Main] Inference time (prefill): 53.43 ms
[INFO][2026-07-23 19:52:11.771][infer.py:127] qwen3_8b decode average inference time cost is 41.20 ms

生成文本:
The output of an attention function is a **weighted sum of the value vectors**, where the weights are determined by the similarity between the **query vector** and each **key vector** in the set of key-value pairs.

### Mathematically, the attention function can be described as:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Where:
- $ Q $ is the **query matrix** (shape: $ [n \times d_k] $)
- $ K $ is the **key matrix** (shape: $ [m \times d_k] $)
- $ V $ is the **value matrix** (shape: $ [m \times d_v] $)
- $ d_k $ is the dimension of the key

### 7.3 查看 Profiling 结果

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv`中展示了模型调用的所有算子的统计信息，可快速确认自定义算子 QmmCustom 的调用情况。`kernel_details.csv`中展示了详细的接口调用信息，可自行查看筛选自己关注的算子的性能。

通过如下操作统计 `kernel_details.csv` 中的 QmmCustom 算子耗时，可自行对比自定义算子替换前后的性能差异。

In [12]:
import pandas as pd
from pathlib import Path

RES_DIR = RECIPE_ROOT / "models/qwen/res"
case_name = "qwen3_8b_" + YAML_A8W8_CUSTOM_PROFILER.stem
case_dirs = sorted(RES_DIR.glob(f"*/{case_name}"))
assert case_dirs, f"未找到模型 Profiler 结果: {case_name}"
latest_result = case_dirs[-1]
print("最新模型 Profiler 结果:", latest_result)

def find_kernel_csv(phase):
    matches = sorted((latest_result / "prof" / phase).rglob("kernel_details.csv"))
    return matches[-1] if matches else None

def select_qmm(df):
    names = df.get("Name", pd.Series("", index=df.index)).fillna("")
    types = df.get("Type", pd.Series("", index=df.index)).fillna("")
    return df[
        names.str.contains("qmm_custom_kernel", case=False)
        | types.str.contains("QmmCustom", case=False)
    ].copy()

rows = []
for phase in ("prefill", "decode"):
    csv_path = find_kernel_csv(phase)
    assert csv_path, f"{phase} kernel_details.csv 未找到"
    phase_df = pd.read_csv(csv_path)
    qmm = select_qmm(phase_df)
    assert not qmm.empty, f"{phase} 中未找到 QmmCustom"
    duration_column = next(
        column for column in qmm.columns
        if column.lower().startswith("duration")
    )
    rows.append({
        "Phase": phase.capitalize(),
        "Calls": len(qmm),
        "Avg Duration(us)": float(qmm[duration_column].mean()),
        "Total Duration(us)": float(qmm[duration_column].sum()),
        "Min Duration(us)": float(qmm[duration_column].min()),
        "Max Duration(us)": float(qmm[duration_column].max()),
    })

model_profile_summary = pd.DataFrame(rows)
display(model_profile_summary)


最新模型 Profiler 结果: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_submission_profiler


,Phase,Calls,Avg Duration(us),Total Duration(us),Min Duration(us),Max Duration(us)
0,Prefill,144,63.676847,9169.466,31.58,111.643
1,Decode,432,46.819576,20226.057,21.30,98.042


## 8 提交要求

> 完整的提交说明参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

1. 提交完整的 Jupyter Notebook（.ipynb），**保留所有单元格输出**。
2. 提交前依次运行所有单元格，确保：
   - 自定义算子编译成功；
   - 算子功能测试通过；
   - 性能数据收集成功。
3. **文件命名**：`gitcode账号_result.ipynb`。
4. **允许并且鼓励**使用 AI 工具辅助，但须独立思考并在报告中简述（解决了什么疑惑、是否引入新 bug）。
5. **不得抄袭。**

---
### 小结

本章完成了 QmmCustom 算子的开发、编译和单算子功能性能测试，并将该自定义算子接入了 Qwen3-8B 量化模型，验证功能并观测了其在模型中的性能。


### 附录
实践相关参考材料链接：
- [Ascend C 高阶api文档](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/%E9%AB%98%E9%98%B6API/%E9%AB%98%E9%98%B6API.md)
- [昇腾社区文档](https://www.hiascend.com/zh/document)
- [Ascend C 算子开发系列教程](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development)
- [Ascend C 算子功能调试](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development/07_Troubleshooting)
- [Matmul最佳实践样例](https://gitcode.com/cann/asc-devkit/blob/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance/README.md)
- [CANN内置量化矩阵乘算子实现](https://gitcode.com/cann/ops-nn/tree/master/matmul/quant_batch_matmul_v3)